In [1]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import urllib.request
import io
from PIL import Image
from dotenv import load_dotenv
import os
load_dotenv()

/home/misango/anaconda3/envs/research_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


True

In [2]:
ee.Authenticate()
ee.Initialize(project=os.getenv('GOOGLE_PROJECT_ID'))

In [3]:
def analyze_smelter_thermal(polygon_coords, start_date, end_date):
    ee.Initialize()
    roi = ee.Geometry.Polygon(polygon_coords)
    
    # Filter collection and map the Normalized Hotspot Index (NHI)
    collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(roi)
                  .filterDate(start_date, end_date)
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))
    
    def calculate_nhi(image):
        swir1 = image.select('B11')
        swir2 = image.select('B12')
        nhi = image.expression(
            '(B12 - B11) / (B12 + B11)', 
            {'B11': swir1, 'B12': swir2}
        ).rename('NHI')
        # Mask out anything that isn't a high thermal emission hotspot
        hotspot = nhi.gt(0.1) 
        return image.addBands(nhi).updateMask(hotspot)

    processed = collection.map(calculate_nhi)
    # Reduce region to get count of active hotspot pixels over time
    stats = processed.select('NHI').map(lambda img: ee.Feature(None, {
        'date': img.date().format('YYYY-MM-DD'),
        'hotspot_pixels': img.reduceRegion(ee.Reducer.count(), roi, scale=20).get('NHI')
    }))
    return stats.getInfo()